<a href="https://colab.research.google.com/github/Ali-Hamza-developer/NLP/blob/main/06_parts_of_speech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part of Speech (POS) Tagging in spaCy

## 1. What is POS Tagging? (Easy Explanation)

POS tagging means labelling every word with **what kind of word it is** — noun, verb, adjective, etc.

```
"Elon flew to mars"
   |      |    |   |
 PROPN   VERB ADP NOUN
(name)  (action)(link)(thing)
```

spaCy gives you TWO levels of detail for this:
- **`token.pos_`** → the simple, universal tag (NOUN, VERB, ADJ, PROPN...)
- **`token.tag_`** → a more detailed, fine-grained tag (e.g. VBD = past tense verb, VBZ = present tense verb)

Think of `pos_` as the broad category, and `tag_` as the exact grammatical detail inside that category.

## 2. Basic POS Tags — `token.pos_`

In [12]:
import spacy
nlp = spacy.load("en_core_web_sm")

doc = nlp("Elon flew to mars yesterday. He carried biryani masala with him")

for token in doc:
    print(token, " | ", token.pos_, " | ", spacy.explain(token.pos_))   # explain() gives the full readable name


Elon  |  PROPN  |  proper noun
flew  |  VERB  |  verb
to  |  ADP  |  adposition
mars  |  NOUN  |  noun
yesterday  |  NOUN  |  noun
.  |  PUNCT  |  punctuation
He  |  PRON  |  pronoun
carried  |  VERB  |  verb
biryani  |  ADJ  |  adjective
masala  |  NOUN  |  noun
with  |  ADP  |  adposition
him  |  PRON  |  pronoun



Full list of all POS categories: https://v2.spacy.io/api/annotation

In [13]:
doc = nlp("Wow! Dr. Strange made 265 million $ on the very first day")

for token in doc:
    print(token, " | ", token.pos_, " | ", spacy.explain(token.pos_))


Wow  |  INTJ  |  interjection
!  |  PUNCT  |  punctuation
Dr.  |  PROPN  |  proper noun
Strange  |  PROPN  |  proper noun
made  |  VERB  |  verb
265  |  NUM  |  numeral
million  |  NUM  |  numeral
$  |  NUM  |  numeral
on  |  ADP  |  adposition
the  |  DET  |  determiner
very  |  ADV  |  adverb
first  |  ADJ  |  adjective
day  |  NOUN  |  noun


## 3. Fine-Grained Tags — `token.tag_`

This gives extra grammar detail on top of `pos_` (e.g. is it past tense or present tense, singular or plural).

In [14]:
doc = nlp("Wow! Dr. Strange made 265 million $ on the very first day")

for token in doc:
    print(token, " | ", token.pos_, " | ", spacy.explain(token.pos_), " | ", token.tag_, " | ", spacy.explain(token.tag_))


Wow  |  INTJ  |  interjection  |  UH  |  interjection
!  |  PUNCT  |  punctuation  |  .  |  punctuation mark, sentence closer
Dr.  |  PROPN  |  proper noun  |  NNP  |  noun, proper singular
Strange  |  PROPN  |  proper noun  |  NNP  |  noun, proper singular
made  |  VERB  |  verb  |  VBD  |  verb, past tense
265  |  NUM  |  numeral  |  CD  |  cardinal number
million  |  NUM  |  numeral  |  CD  |  cardinal number
$  |  NUM  |  numeral  |  CD  |  cardinal number
on  |  ADP  |  adposition  |  IN  |  conjunction, subordinating or preposition
the  |  DET  |  determiner  |  DT  |  determiner
very  |  ADV  |  adverb  |  RB  |  adverb
first  |  ADJ  |  adjective  |  JJ  |  adjective (English), other noun-modifier (Chinese)
day  |  NOUN  |  noun  |  NN  |  noun, singular or mass


## 4. `tag_` Can Tell Past vs Present Tense — Something `pos_` Alone Can't

In [15]:
doc = nlp("He quits the job")   # present tense
print(doc[1].text, "|", doc[1].tag_, "|", spacy.explain(doc[1].tag_))


quits | VBZ | verb, 3rd person singular present


**Output:** `quits | VBZ | verb, 3rd person singular present`

In [16]:
doc = nlp("he quit the job")   # past tense
print(doc[1].text, "|", doc[1].tag_, "|", spacy.explain(doc[1].tag_))


quit | VBD | verb, past tense


**Output:** `quit | VBD | verb, past tense`

Same word family ("quit"), but `tag_` correctly tells present (VBZ) from past (VBD). This is why `tag_` is useful when tense/number matters.

## 5. Cleaning Text — Removing SPACE, PUNCT, X Tokens

When doing real text analysis, you usually don't want punctuation, extra whitespace, or junk (`X`) tokens cluttering your results. Filter them out using `pos_`.

In [17]:
earnings_text = """Microsoft Corp. today announced the following results for the quarter ended December 31, 2021, as compared to the corresponding period of last fiscal year:

·         Revenue was $51.7 billion and increased 20%
·         Operating income was $22.2 billion and increased 24%
·         Net income was $18.8 billion and increased 21%
·         Diluted earnings per share was $2.48 and increased 22%
"Digital technology is the most malleable resource at the world’s disposal to overcome constraints and reimagine everyday work and life," said Satya Nadella, chairman and chief executive officer of Microsoft. "As tech as a percentage of global GDP continues to increase, we are innovating and investing across diverse and growing markets, with a common underlying technology stack and an operating model that reinforces a common strategy, culture, and sense of purpose."
"Solid commercial execution, represented by strong bookings growth driven by long-term Azure commitments, increased Microsoft Cloud revenue to $22.1 billion, up 32% year over year" said Amy Hood, executive vice president and chief financial officer of Microsoft."""

doc = nlp(earnings_text)

filtered_tokens = []
for token in doc:
    if token.pos_ not in ["SPACE", "PUNCT", "X"]:   # skip junk tokens
        filtered_tokens.append(token)

filtered_tokens[:10]


[Microsoft,
 Corp.,
 today,
 announced,
 the,
 following,
 results,
 for,
 the,
 quarter]

## 6. Counting POS Tags in a Document — `doc.count_by()`

Useful for quickly seeing how many nouns, verbs, etc. exist in a text (great for text analysis / summarizing style).

In [18]:
count = doc.count_by(spacy.attrs.POS)
count   # returns tag IDs (numbers) mapped to their frequency


{96: 15,
 92: 45,
 100: 23,
 90: 9,
 85: 16,
 93: 16,
 97: 27,
 98: 1,
 84: 20,
 103: 10,
 87: 6,
 99: 5,
 89: 12,
 86: 3,
 94: 3,
 95: 2}

In [19]:
# convert the number IDs into readable tag names using doc.vocab
for k, v in count.items():
    print(doc.vocab[k].text, "|", v)


PROPN | 15
NOUN | 45
VERB | 23
DET | 9
ADP | 16
NUM | 16
PUNCT | 27
SCONJ | 1
ADJ | 20
SPACE | 10
AUX | 6
SYM | 5
CCONJ | 12
ADV | 3
PART | 3
PRON | 2


---
## Quick Cheat Sheet

| Task | Code |
|---|---|
| Simple POS tag | `token.pos_` |
| Detailed grammar tag | `token.tag_` |
| Human-readable meaning | `spacy.explain(tag)` |
| Remove junk tokens | `if token.pos_ not in ["SPACE","PUNCT","X"]` |
| Count all POS tags | `doc.count_by(spacy.attrs.POS)` |
| Convert tag ID -> name | `doc.vocab[id].text` |

---

## Exercise — Extract NOUN & NUM Tokens from a News Article + Count All POS Tags

**Task:** Read a news article from a text file (`news_story.txt`), extract all NOUN tokens and all NUM tokens separately, then print a full count of every POS tag found in the article.

> **Note:** This exercise needs a text file named `news_story.txt` in the same folder as this notebook (a short news article works fine — grab any news paragraph and save it as `news_story.txt`).

In [20]:
sample_text = """Inflation rose again in April, continuing a climb that has pushed consumers to the brink and is threatening the economic expansion, the Bureau of Labor Statistics reported Wednesday.

The consumer price index, a broad-based measure of prices for goods and services, increased 8.3% from a year ago, higher than the Dow Jones estimate for an 8.1% gain. That represented a slight ease from March's peak but was still close to the highest level since the summer of 1982.

Removing volatile food and energy prices, so-called core CPI still rose 6.2%, more than the 6% expected.

The month-over-month gains also were higher than expected: 0.3% on headline CPI versus the 0.2% estimate, and a 0.6% increase for core, against the outlook for a 0.4% gain.

The Bureau of Labor Statistics said price increases for shelter, food, airline fares and new vehicles were the largest contributors to the monthly increase. Energy prices fell 2.7% for the month but remain up 30.3% year over year."""

with open("news_story.txt", "w") as f:
    f.write(sample_text)

print("news_story.txt created!")

news_story.txt created!


In [21]:
# step 1: read the news article
with open("news_story.txt", "r") as f:
    news_text = f.read()

news_text[:500]   # preview first 500 characters


"Inflation rose again in April, continuing a climb that has pushed consumers to the brink and is threatening the economic expansion, the Bureau of Labor Statistics reported Wednesday.\n\nThe consumer price index, a broad-based measure of prices for goods and services, increased 8.3% from a year ago, higher than the Dow Jones estimate for an 8.1% gain. That represented a slight ease from March's peak but was still close to the highest level since the summer of 1982.\n\nRemoving volatile food and energ"

**Expected preview output (from the original AP inflation article used in this exercise):**
```
'Inflation rose again in April, continuing a climb that has pushed consumers to the brink and is threatening the economic expansion, the Bureau of Labor Statistics reported Wednesday.\n\nThe consumer price index, a broad-based measure of prices for goods and services, increased 8.3% from a year ago, higher than the Dow Jones estimate for an 8.1% gain. ...'
```

In [22]:
# step 2: process text and separate NOUN and NUM tokens
doc = nlp(news_text)

numeral_tokens = []
noun_tokens = []

for token in doc:
    if token.pos_ == "NOUN":
        noun_tokens.append(token)
    elif token.pos_ == "NUM":
        numeral_tokens.append(token)


In [23]:
numeral_tokens[:10]


[8.3, 8.1, 1982, 6.2, 6, 0.3, 0.2, 0.6, 0.4, 2.7]

**Expected Output:** `[8.3, 8.1, 1982, 6.2, 6, 0.3, 0.2, 0.6, 0.4, 0.1]`

In [24]:
noun_tokens[:10]


[Inflation,
 climb,
 consumers,
 brink,
 expansion,
 consumer,
 price,
 index,
 measure,
 prices]

**Expected Output:** `[Inflation, climb, consumers, brink, expansion, consumer, price, index, measure, prices]`

In [25]:
# step 3: count every POS tag in the whole article
count = doc.count_by(spacy.attrs.POS)

for k, v in count.items():
    print(doc.vocab[k].text, "|", v)


NOUN | 56
VERB | 16
ADV | 8
ADP | 23
PROPN | 13
PUNCT | 23
DET | 22
PRON | 2
AUX | 5
CCONJ | 7
ADJ | 11
SPACE | 4
NUM | 11
PART | 1
SCONJ | 2
